![title](imagens/projeto22.jpg)

# Projeto 22 - Fitness Tracker Data (Dados Rastreados de Atividades Físicas)
### Análise de Dados

**Sobre o Conjunto de Dados**

Este conjunto de dados foi gerado por respondentes de uma pesquisa distribuída via Amazon Mechanical Turk entre 03.12.2016 a 05.12.2016. Trinta usuários elegíveis do Fitbit consentiram com a submissão de dados pessoais do rastreador, incluindo saída em nível de minuto para atividade física, frequência cardíaca e monitoramento do sono. Relatórios individuais podem ser analisados por ID de sessão de exportação (coluna A) ou carimbo de data/hora (coluna B). A variação entre as saídas representa o uso de diferentes tipos de rastreadores Fitbit e comportamentos/preferências individuais de rastreament

Fonte: Kaggle (https://www.kaggle.com/datasets/arashnic/fitbit/data).
Dentro deste dataset existem muitas informações e arquivos. Vamos trabalhar apenas com o **dailyActivity_merged.csv** de 4.12.16-5.12.1 Este arquivo está disponível no subdiretório ``dados``...


In [ ]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python usada neste Projeto no Jupyter Notebook:', python_version())

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
# !pip install -q -U watermark


### Importando as Bibliotecs / Pacotes necessários

In [ ]:
# Importação dos pacotes básicos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Importação do pacote datetime para trabalhar com dados tipo datetime
import datetime as dt


# para evitar mensagens de alerta/warnings.
import warnings
warnings.filterwarnings("ignore")

# Carregar o módulo de funções para limpeza de dados
from limpeza_dados import *

In [ ]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "pyPRO - Seja um Profissional Python!" --iversions

In [ ]:
# Abrindo o dataset
df = pd.read_csv('dados/dailyActivity_merged.csv')

In [ ]:
# Exibindo o cabeçalho dos dados
df.head(10)

## Preprocessamento dos dados

In [ ]:
# Verificando a quantidade de número de dados únicos no dataset
# verificação pelo Id
df['Id'].nunique()

Vamos trabalhar com o total de passos e minutos e qual a relação com as calorias totais. Vamos remover as colunas irrelevantes dos dados.

In [ ]:
# Vamos fazer a filtragem apenas dos dados relevantes o que tornará mais simples e fácil a tarefa de análise dos dados
data = [
    'Id','ActivityDate','TotalSteps','VeryActiveMinutes',
    'FairlyActiveMinutes','LightlyActiveMinutes',
    'SedentaryMinutes','Calories'
]
df = df[data]
df.head(10)

Vamos dar mais siginficado para os dados... fazendo a engenharia dos dados:

Renomeando a coluna 'ActivityDate' em 'Date'

In [ ]:
df.rename(columns={'ActivityDate':'Date'},inplace=True)
df.head(10)

Vamos criar uma coluna com o total de minutos de todas as atividades realizadas

In [ ]:
df['TotalMinutes'] = df['FairlyActiveMinutes'] + df['LightlyActiveMinutes'] + df['SedentaryMinutes']+ df['VeryActiveMinutes']
df.head(10)

Criando uma outra coluna do total de horas de todas as atividades

In [ ]:
df['TotalHours'] = np.round(df['TotalMinutes']/60)
df.head(10)

## Mostrando os dados após essas transformações...

Verificando os valores, para encontrar nulos e não checados.

In [ ]:
# Verificando os valores
df.info()

In [ ]:
# Mudando a coluna 'Date' para datetime
df['Date'] = pd.to_datetime(df['Date'])
# verificando novamente
df.info()

In [ ]:
# Vamos incluir uma nova coluna que contem o dia da semana (DayOfWeek)
df['DayOfWeek'] = df['Date'].dt.day_name()

In [ ]:
# Exibindo os dados
df.head(10)

In [ ]:
# verificando se existe algum valor nulo
calcular_porcentagem_valores_ausentes(df)

Muito bom! Não existem valores nulos no dataset

Vamos ver agora se existem valores duplicados

In [ ]:
df.duplicated().sum()

Também não existem valores duplicados.

## Análise e visualização dos dados

In [ ]:
# Descrição dos dados
df.describe()

Podemos observar pelas colunas TotalSteps, VeryActiveMinutes, FairlyActiveMinutes e LightlyActiveMinutes que a maioria das pessoas não pratica esportes devido à grande diferença entre o total de passos e os passos ativos.

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df['DayOfWeek'], bins=7, color='lightskyblue', width=0.6)
plt.xticks(rotation = 90)
plt.grid()
plt.title("Freq by day of week")
plt.xlabel('Days Of Week')
plt.ylabel('Frequency')
plt.show()

Observamos que as pessoas são muito ativas às terças, quartas e quintas-feiras, então podemos enviar mensagens de motivação para as pessoas nos outros dias.

In [ ]:
# Para selecionar apenas os valores numericos
numerical_columns = df.select_dtypes(exclude=object).columns.tolist()

In [ ]:
# Vamos mostrar a correlação entre essas colunas de dados numéricos
sns.heatmap(df[numerical_columns].corr())

A partir do mapa de calor, podemos observar que as colunas TotalSteps e VeryActiveMinutes têm a maior influência na coluna de Calorias.

In [ ]:
# Vamos então visualizar a relação entre a coluna TotalSteps e a coluna Calories
plt.figure(figsize=(8,6))
plt.scatter(df['TotalSteps'],df['Calories'],c = df['Calories'])
mediansteps =  7405
medianCalory = 2134
plt.axhline(medianCalory, color = 'blue', label = "Median of Calories")
plt.axvline(mediansteps, color = 'red', label = "Median of steps")
plt.title("Calories by steps")
plt.xlabel('Steps')
plt.ylabel('Calories')
plt.legend()
plt.show()

In [ ]:
# Vamos visualizar a relação entre a coluna TotalHours com a coluna Calories
plt.figure(figsize=(8,6))
plt.scatter(df['TotalHours'],df['Calories'],c = df['Calories'])
medianHours =  24
medianCalory = 2134
plt.axhline(medianCalory, color = 'blue', label = "Median of Calories")
plt.axvline(medianHours, color = 'red', label = "Median of Hours")
plt.title("Calories by Hours")
plt.xlabel('Hours')
plt.ylabel('Calories')
plt.legend()
plt.show()

Observamos que existe uma relação fraca entre eles. O que pode explicar isso é o baixo número de minutos ativos.

In [ ]:
# Visualizando o percentual de cada coluna com essas outras colunas {VeryActiveMinutes, FairlyActiveMinutes, LightlyActiveMinutes, SedentaryMinutes}
FairlyActiveMinutes = df['FairlyActiveMinutes'].sum()
VeryActiveMinutes = df['VeryActiveMinutes'].sum()
LightlyActiveMinutes = df['LightlyActiveMinutes'].sum()
SedentaryMinutes = df['SedentaryMinutes'].sum()

minuts = [FairlyActiveMinutes,VeryActiveMinutes,LightlyActiveMinutes,SedentaryMinutes]
label = ['FairlyActiveMinutes','VeryActiveMinutes','LightlyActiveMinutes','SedentaryMinutes']

plt.pie(minuts,labels=label,autopct='%1.4f%%',explode=[0,0,0,0.2])
plt.show()

Podemos dizer que 81 por cento dos usuários utilizam o programa para calcular as calorias queimadas em atividades diárias normais, e também são muito ativos no meio e no final da semana.

## Fim do Projeto 22